In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
!pip install chemprop==2.2.2

# Etapas Preliminares/Preliminary stetps

In [ ]:
df=pd.read_csv('/input/folder/yourtable.csv', sep=';', low_memory=False)

In [ ]:
df2 = df[df["Standard Value"].notna() & df["Smiles"].notna()]

In [ ]:
len(df2.Smiles.unique())

In [ ]:
df2_nr = df2.drop_duplicates(['Smiles','Molecule ChEMBL ID'])
df2_nr = df2_nr[df2_nr['Standard Relation'] == "'='" ]
df2_nr = df2_nr[(df2_nr['Comment'].isna())]
df2_nr

In [ ]:
selection = ['Molecule ChEMBL ID','Smiles','Standard Value']
df3 = df2_nr[selection]
df3

In [ ]:
df3.to_csv('mol_data_preprocessed.csv', index=False)

# Labeling

In [ ]:
df4 = pd.read_csv('mol_data_preprocessed.csv')

In [ ]:
bioactivity_threshold = []
for i in df4["Standard Value"]:
    if float(i) > 1000.0:
        bioactivity_threshold.append("0")  # inactive
    elif float(i) <= 1000.0:
        bioactivity_threshold.append("1")  # active

In [ ]:
bioactivity_class = pd.Series(bioactivity_threshold, name='class')
df5 = pd.concat([df4, bioactivity_class], axis=1)
df5

In [ ]:
df5.rename(columns={'Standard Value': 'Standard_value'}, inplace=True)

In [ ]:
df5.to_csv('mol_data_curated.csv', index=False)

# Otimização de hiperparametros // Hyperparameter optimization

In [ ]:
!pip install 'ray==2.48.*'

In [ ]:
!chemprop hpopt --data-path mol_data_curated.csv --task-type classification --search-parameter-keywords basic --smiles-columns Smiles --target-columns class --split-type scaffold_balanced --hpopt-save-dir results_hpt_f --raytune-use-gpu --raytune-num-gpus 2 --logfile

# Treino CLI// Training using CLI

In [ ]:
!chemprop train --data-path /input/folder/mol_data_curated.csv --config-path /kaggle/working/results_hpt_f/best_config.toml --task-type classification --metrics {roc,f1,accuracy,prc} --split-type scaffold_balanced  --num-replicates 5 --smiles-columns Smiles --target-columns class --epochs 200 --patience 20 --accelerator gpu -n 2 -b 128 --logfile